# <b>Matplotlib로 심혈관 CSV 그리기</b>

- 목표: `matplotlib`로 막대그래프, 히스토그램, 박스플롯을 그립니다.
- 연결: 메인 실습의 시각화 파트로 들어가기 전에 그래프의 기본 구조를 익히는 브리지입니다.


In [1]:
# Shared lecture path bootstrap
from pathlib import Path
from collections import deque
import os
import sys
import unicodedata

try:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

SEARCH_ROOTS = (
    Path("/content/drive/MyDrive/Colab Notebooks"),
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
    Path.home(),
)
PROJECT_NAMES = {"00_의료 데이터 분석"}

def _norm(text: str) -> str:
    return unicodedata.normalize("NFC", text)

def _is_project_root(path: Path) -> bool:
    return (path / "00_shared" / "utils" / "lecture_paths.py").exists()

def _find_project_root() -> Path:
    override = os.environ.get("PROJECT_ROOT")
    if override:
        root = Path(override).expanduser().resolve()
        if _is_project_root(root):
            return root
        raise FileNotFoundError("환경변수 PROJECT_ROOT가 올바른 강의 루트를 가리키지 않습니다.")

    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents]:
        if _is_project_root(root):
            return root

    target_names = {_norm(name) for name in PROJECT_NAMES}
    for base in SEARCH_ROOTS:
        if not base.exists():
            continue
        queue = deque([(base.resolve(), 0)])
        seen = set()
        while queue:
            current, depth = queue.popleft()
            if current in seen:
                continue
            seen.add(current)

            if _norm(current.name) in target_names and _is_project_root(current):
                return current
            if depth >= 4:
                continue

            try:
                children = [
                    child for child in current.iterdir()
                    if child.is_dir() and not child.name.startswith(".") and child.name not in {"__pycache__", ".ipynb_checkpoints"}
                ]
            except OSError:
                continue

            children.sort(key=lambda child: (_norm(child.name) not in target_names, child.name))
            queue.extend((child.resolve(), depth + 1) for child in children)

    raise FileNotFoundError(
        "PROJECT_ROOT를 찾지 못했습니다. Colab에서는 os.environ['PROJECT_ROOT']를 먼저 지정하세요."
    )

def _resolve_day_dir(name: str) -> Path:
    for candidate in {name, unicodedata.normalize("NFC", name), unicodedata.normalize("NFD", name)}:
        path = PROJECT_ROOT / candidate
        if path.exists():
            return path
    return PROJECT_ROOT / name

PROJECT_ROOT = _find_project_root()
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
UTILS_ROOT = PROJECT_ROOT / "00_shared" / "utils"
if str(UTILS_ROOT) not in sys.path:
    sys.path.append(str(UTILS_ROOT))

from lecture_paths import DATA_ROOT, MODEL_ROOT, ensure_dir

DAY_DIR = _resolve_day_dir('00-Data-Analysis')
DAY_OUTPUT_ROOT = ensure_dir(DAY_DIR / "99_실행산출물")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"MODEL_ROOT: {MODEL_ROOT}")
print(f"DAY_OUTPUT_ROOT: {DAY_OUTPUT_ROOT}")

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/00_Project/00_의료 데이터 분석
DATA_ROOT: /content/drive/MyDrive/00_Project/00_의료 데이터 분석/00_shared/datasets
MODEL_ROOT: /content/drive/MyDrive/00_Project/00_의료 데이터 분석/00_shared/models
DAY_OUTPUT_ROOT: /content/drive/MyDrive/00_Project/00_의료 데이터 분석/00-Data-Analysis/99_실행산출물


## 실행 가이드

- 이 노트북은 `matplotlib`만 사용해 그래프를 그립니다.
- 그래프 제목, 축 이름, 색상은 한두 군데만 직접 바꿔보면 충분합니다.


## AI 도구 활용 체크포인트

- 현재 그래프가 무엇을 보여주는지 비전공자 기준으로 설명하게 한 뒤, 실제 축과 범례가 맞는지 직접 확인합니다.
- 막대그래프와 히스토그램의 차이를 한 문장으로 설명하게 해보세요.


In [2]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = DATA_ROOT / "heart_disease" / "heart_disease.csv"
df = pd.read_csv(csv_path, keep_default_na=False, na_values=[""])
print(f"shape: {df.shape}")

shape: (10000, 21)


# Part 1. 막대그래프와 히스토그램

## 🎯 학습 목표
- 범주 분포를 막대그래프로 본다.
- 숫자 분포를 히스토그램으로 본다.


In [4]:
target_counts = df["Heart Disease Status"].value_counts()
print(target_counts)



Heart Disease Status
No     8000
Yes    2000
Name: count, dtype: int64


In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df["Cholesterol Level"].dropna(), bins=20, color="#7aa6c2", edgecolor="black")
plt.title("Cholesterol Level Histogram")
plt.xlabel("Cholesterol Level")
plt.ylabel("Frequency")
plt.show()

# Part 2. 박스플롯과 직접 바꿔보기

## 🎯 학습 목표
- 두 집단의 분포 차이를 박스플롯으로 본다.
- 제목, 축, 변수만 살짝 바꿔본다.


## 📝 TODO

In [ ]:

# plt.figure(figsize=(5, 4))
# plt.bar(target_counts.index, target_counts.values, color=["#7aa6c2", "#e28f83"])
# plt.title("Heart Disease Status Count")
# plt.xlabel("Heart Disease Status")
# plt.ylabel("Count")
# plt.show()

In [ ]:
# BMI 컬럼으로 히스토그램을 하나 더 그려보세요. 제목과 x축 이름도 함께 넣어보세요.

In [ ]:
no_age = df.loc[df["Heart Disease Status"] == "No", "Age"].dropna()
yes_age = df.loc[df["Heart Disease Status"] == "Yes", "Age"].dropna()

plt.figure(figsize=(6, 4))
plt.boxplot([no_age, yes_age], labels=["No", "Yes"])
plt.title("Age by Heart Disease Status")
plt.xlabel("Heart Disease Status")
plt.ylabel("Age")
plt.show()

## 의료 해석 질문

- 막대그래프에서 타깃 분포가 심하게 치우쳐 보이나요?
- 박스플롯 기준으로 어느 집단에서 나이 범위가 더 넓어 보이나요?
